# ETGNN + Matched GAT Development Technical Pilot Benchmark

**Protocol:** Dated Frozen Protocol (v2.1)  
**Target Hardware:** Google Colab GPU (T4 / V100 / A100)  
**Objective:** Measure actual hardware execution latency (ms) per seed, CPU peak RSS, and GPU peak allocated VRAM for GNNExplainer (50 epochs, seeds 42, 43, 44), 30 random deletions, degree ranking, and single-edge occlusion on the **4 saved development validation nodes (steps 33–36)**.  

> [!IMPORTANT]
> **Scope Disclosure:** This benchmark is executed strictly on 4 development-period nodes to profile hardware latency and memory overhead. **It is NOT a matched-pair effect-variance pilot**, and development-period outcomes will not be used to estimate effect sizes or stopping rules.

## 1. Inspect Hardware Environment
Verify that a GPU runtime is active.

In [15]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name:', torch.cuda.get_device_name(0))
    print('VRAM Total (GB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
else:
    print('WARNING: Running on CPU. Please switch to GPU in Runtime -> Change runtime type.')

Tue Oct 20 12:47:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Load the Execution Bundle

The bundle archive `ETGNN_Technical_Pilot_Bundle.zip` is **43.5 MB** (43,534,835 bytes).

> [!WARNING]
> **Avoid Drag-and-Drop Truncation:** In Google Colab, dragging and dropping large files into the left sidebar file browser often truncates the upload at 10–12 MB due to browser WebSocket timeouts, producing a corrupted `BadZipFile`.
>
> **Use one of the two reliable methods below:**
> - **Option A (Recommended & Fastest):** Mount Google Drive (upload `ETGNN_Technical_Pilot_Bundle.zip` to your Google Drive `MyDrive/` first, which never truncates).
> - **Option B:** Use the Colab `files.upload()` widget below, which streams with an active progress bar.

### Option A: Mount Google Drive (Recommended)

In [16]:
import os, shutil

bundle_path = None
# Check if already uploaded directly to /content/
if os.path.exists('/content/ETGNN_Technical_Pilot_Bundle.zip'):
    sz = os.path.getsize('/content/ETGNN_Technical_Pilot_Bundle.zip')
    if sz >= 40000000:
        bundle_path = '/content/ETGNN_Technical_Pilot_Bundle.zip'
        print(f'Found valid complete bundle in /content/ ({sz:,} bytes).')
    else:
        print(f'Detected incomplete upload in /content/ ({sz:,} / 43,534,835 bytes).')

if bundle_path is None:
    print('Mounting Google Drive to look for bundle...')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_candidates = [
            '/content/drive/MyDrive/ETGNN_Technical_Pilot_Bundle.zip',
            '/content/drive/MyDrive/Colab Notebooks/ETGNN_Technical_Pilot_Bundle.zip',
        ]
        for c in drive_candidates:
            if os.path.exists(c):
                sz = os.path.getsize(c)
                if sz >= 40000000:
                    bundle_path = c
                    print(f'Found complete bundle in Google Drive: {c} ({sz:,} bytes)')
                    break
    except Exception as e:
        print(f'Drive mount skipped or failed: {e}')

if bundle_path:
    print('SUCCESS: Bundle located at:', bundle_path)
else:
    print('Bundle not found in Drive. Please run Option B below to upload via progress widget.')

Found valid complete bundle in /content/ (43,534,835 bytes).
SUCCESS: Bundle located at: /content/ETGNN_Technical_Pilot_Bundle.zip


### Option B: Direct Upload via Colab Widget (with Progress Bar)
Run this cell only if Option A was not used.

In [17]:
if bundle_path is None:
    from google.colab import files
    print('Please select ETGNN_Technical_Pilot_Bundle.zip from your computer:')
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            bundle_path = os.path.abspath(fn)
            print(f'Uploaded: {bundle_path} ({os.path.getsize(bundle_path):,} bytes)')
            break

## 3. Verify Size & Extract Archive

In [18]:
import os, zipfile

EXPECTED_BYTES = 43534835
if not bundle_path or not os.path.exists(bundle_path):
    raise FileNotFoundError('ETGNN_Technical_Pilot_Bundle.zip has not been uploaded yet.')

actual_bytes = os.path.getsize(bundle_path)
print(f'Archive location: {bundle_path}')
print(f'Archive size:     {actual_bytes:,} bytes (Expected: {EXPECTED_BYTES:,} bytes)')

if actual_bytes < 40000000:
    raise RuntimeError(
        f'TRUNCATED FILE ERROR: Archive is only {actual_bytes:,} bytes, which is missing data! '
        f'The complete file must be {EXPECTED_BYTES:,} bytes. '
        'Please upload the bundle to your Google Drive (MyDrive) and mount it via Option A, '
        'or use Option B files.upload() to re-upload completely.'
    )

work_dir = '/content/pilot_workdir'
if os.path.exists(work_dir):
    import shutil
    shutil.rmtree(work_dir)
os.makedirs(work_dir, exist_ok=True)

print(f'Extracting to {work_dir}...')
with zipfile.ZipFile(bundle_path, 'r') as z:
    z.extractall(work_dir)

os.chdir(work_dir)
print('SUCCESS: Extraction complete.')
!ls -lh

Archive location: /content/ETGNN_Technical_Pilot_Bundle.zip
Archive size:     43,534,835 bytes (Expected: 43,534,835 bytes)
Extracting to /content/pilot_workdir...
SUCCESS: Extraction complete.
total 20K
drwxr-xr-x 3 root root 4.0K Oct 20 12:47 data
drwxr-xr-x 3 root root 4.0K Oct 20 12:47 experiments
-rw-r--r-- 1 root root 1.3K Oct 20 12:47 manifest.sha256
drwxr-xr-x 3 root root 4.0K Oct 20 12:47 results
drwxr-xr-x 4 root root 4.0K Oct 20 12:47 src


## 4. Cryptographic SHA-256 Manifest Verification

In [19]:
import hashlib

print('Verifying extracted artifacts against manifest.sha256...')
with open('manifest.sha256', 'r') as f:
    lines = f.readlines()

all_passed = True
for line in lines:
    line = line.strip()
    if not line:
        continue
    expected_hash, sz, rel_path = line.split(maxsplit=2)
    if not os.path.exists(rel_path):
        print(f'FAIL: {rel_path} missing!')
        all_passed = False
        continue
    h = hashlib.sha256(open(rel_path, 'rb').read()).hexdigest()
    if h != expected_hash:
        print(f'FAIL: Hash mismatch for {rel_path}!')
        all_passed = False
    else:
        print(f'OK: {rel_path:<45} ({int(sz):>10,} bytes)')

assert all_passed, 'Artifact manifest verification failed!'
print('\nAll 13 artifacts cryptographically verified.')

Verifying extracted artifacts against manifest.sha256...
OK: src/__init__.py                               (        30 bytes)
OK: src/models/__init__.py                        (       177 bytes)
OK: src/models/etgnn.py                           (       929 bytes)
OK: src/models/matched_gat.py                     (       512 bytes)
OK: src/models/time_encoding.py                   (       494 bytes)
OK: src/evaluation/__init__.py                    (        41 bytes)
OK: src/evaluation/run_technical_pilot_benchmark.py (    18,899 bytes)
OK: data/processed/node_mapping.csv               ( 4,033,371 bytes)
OK: data/processed/validation_graph.pt            (114,002,507 bytes)
OK: results/checkpoints/etgnn.pt                  (    50,929 bytes)
OK: results/checkpoints/matched_gat.pt            (    51,333 bytes)
OK: experiments/matched_c1_c2_audit/technical_pilot_sample.csv (       730 bytes)

All 13 artifacts cryptographically verified.


## 5. Install Dependencies

In [20]:
!pip install -q torch-geometric psutil

## 6. Execute Technical Pilot Benchmark

In [24]:
from pathlib import Path
import ast
import shutil
import importlib
import sys
import os

root = Path('/content/pilot_workdir')
runner = root / 'src/evaluation/run_technical_pilot_benchmark.py'
source = runner.read_text(encoding='utf-8')

# Confirm the replacement variable exists
tree = ast.parse(source)
assigned = {
    n.id for n in ast.walk(tree)
    if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Store)
}

if 'd_c1_gt' not in assigned:
    raise RuntimeError("d_c1_gt assignment not found; stop here.")

old = '"c1_drop_gat": d_c1_gat'
new = '"c1_drop_gat": d_c1_gt'

if old in source:
    backup = runner.with_suffix('.py.before_typo_fix')
    if not backup.exists():
        shutil.copy2(runner, backup)

    corrected = source.replace(old, new)
    ast.parse(corrected)
    runner.write_text(corrected, encoding='utf-8')
    print("Typo fixed.")
elif new in source:
    print("File already contains the fix.")
else:
    raise RuntimeError("Expected line not found; stop here.")

# Reload the corrected module
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import src.evaluation.run_technical_pilot_benchmark as benchmark
importlib.invalidate_caches()
benchmark = importlib.reload(benchmark)

print("Corrected module reloaded. Now run the cell below.")

Typo fixed.
Corrected module reloaded. Now run the cell below.


In [25]:
summary_df, summary_json = benchmark.run_benchmark(
    sample_csv_path='experiments/matched_c1_c2_audit/technical_pilot_sample.csv',
    val_graph_path='data/processed/validation_graph.pt',
    node_mapping_path='data/processed/node_mapping.csv',
    etgnn_ckpt_path='results/checkpoints/etgnn.pt',
    matched_gat_ckpt_path='results/checkpoints/matched_gat.pt',
    output_dir='experiments/technical_pilot_results',
)

  ETGNN + MATCHED GAT DEVELOPMENT TECHNICAL PILOT BENCHMARK
Hardware Device: cuda (CUDA available: True)
GPU Device Name: Tesla T4

--- 1. Checking Inputs & SHA-256 Manifest ---
  technical_pilot_sample.csv     |        730 bytes | SHA256: 5674875923581d8d...
  validation_graph.pt            |  114002507 bytes | SHA256: 261b7ee5951d386b...
  node_mapping.csv               |    4033371 bytes | SHA256: 90ae469bb1449e51...
  etgnn.pt                       |      50929 bytes | SHA256: b2f5535f3b6b1675...
  matched_gat.pt                 |      51333 bytes | SHA256: 1e9b43ea1cc25a26...

--- 2. Asserting 4 Development Nodes ---
  Verified TP (Node 131350, txId 93564753, Step 33, E_cand=2)
  Verified FP (Node 130659, txId 92512070, Step 33, E_cand=118)
  Verified FN (Node 134593, txId 288474786, Step 34, E_cand=2)
  Verified TN (Node 130674, txId 92724813, Step 33, E_cand=6)

--- 3. Loading Models in Eval Mode ---
Computing full-graph inference...

--- 4. Running Benchmark Across 4 Benchmark 

## 7. Display Benchmark Measurements & Runtime Projections

In [27]:
from pathlib import Path
import pandas as pd
from IPython.display import display

filename = 'technical_pilot_benchmark_summary.csv'

# Check both common working directories
paths = [
    Path('/content/pilot_workdir/experiments/technical_pilot_results') / filename,
    Path('/content/experiments/technical_pilot_results') / filename,
]

matches = [p for p in paths if p.is_file()]

# Search extracted project if necessary
if not matches:
    root = Path('/content/pilot_workdir')
    if root.exists():
        matches = list(root.rglob(filename))

if not matches:
    raise FileNotFoundError(
        "Benchmark summary has not been found. "
        "Run the BENCHMARK EXECUTION cell first and send its complete "
        "output/error. The results-display cell cannot create results."
    )

if len(matches) > 1:
    raise RuntimeError(
        "Multiple summaries found. Select the intended run:\n"
        + "\n".join(str(p) for p in matches)
    )

sum_path = matches[0]
print("Reading:", sum_path)

df_res = pd.read_csv(sum_path)
print(f"Summary rows: {len(df_res)}")
display(df_res)

Reading: /content/pilot_workdir/experiments/technical_pilot_results/technical_pilot_benchmark_summary.csv
Summary rows: 4


,node_index,transaction_id,category,time_step,E_cand,k_budget,actual_k_over_E,extract_runtime_ms,gnnexplainer_mean_runtime_ms,random_30_total_runtime_ms,...,occlusion_runtime_ms,peak_rss_mb,peak_vram_mb,c1_target_class,mean_ge_drop_etgnn,mean_ge_drop_gat,mean_rnd_drop_etgnn,mean_rnd_drop_gat,advantage_A_etgnn,advantage_A_gat
0,131350,93564753,TP,33,2,1,0.5000,4.51,327.39,127.78,...,8.85,2018.57,737.28,1,-0.0275,-0.0291,-0.0407,-0.0473,0.0132,0.0182
1,130659,92512070,FP,33,118,30,0.2542,4.94,533.30,142.67,...,271.45,2018.90,737.28,1,0.6130,0.0668,0.0314,-0.0347,0.5816,0.1014
2,134593,288474786,FN,34,2,1,0.5000,5.04,335.60,145.23,...,8.89,2018.93,737.28,0,0.2654,0.1569,0.1610,0.0911,0.1044,0.0658
3,130674,92724813,TN,33,6,2,0.3333,5.07,359.57,133.53,...,18.97,2018.97,737.28,0,0.0850,0.0592,0.0303,0.0117,0.0547,0.0474


## 8. Package Results for Download

In [29]:
!zip -r /content/technical_pilot_results.zip experiments/technical_pilot_results
print('SUCCESS: Results packaged into /content/technical_pilot_results.zip.')

from google.colab import files
try:
    files.download('/content/technical_pilot_results.zip')
    print('Download triggered successfully.')
except Exception as e:
    print(f'Please manual-download from the left folder sidebar if popup is blocked: {e}')

updating: experiments/technical_pilot_results/ (stored 0%)
updating: experiments/technical_pilot_results/input_manifest.json (deflated 54%)
updating: experiments/technical_pilot_results/raw_benchmark_records.csv (deflated 82%)
updating: experiments/technical_pilot_results/pilot_resume_state.json (deflated 21%)
updating: experiments/technical_pilot_results/technical_pilot_benchmark_summary.csv (deflated 46%)
updating: experiments/technical_pilot_results/technical_pilot_summary.json (deflated 40%)
SUCCESS: Results packaged into /content/technical_pilot_results.zip.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered successfully.
